# 07 - Cleaning Eurostat Citizenship Acquisition for Cameroonian Citizens

This notebook cleans Eurostat data on citizenship acquisition by Cameroonian citizens in European destination countries.

Analytical role:

- supports Q3 on post-arrival trajectories
- tracks naturalization-related outcomes across available reference years
- standardizes citizenship acquisition indicators for `post_arrival_master.csv`

Citizenship acquisition is not interpreted as the same phenomenon as first permits, long-term residence or status changes.


## 1. Objective of the Notebook

The objective of this notebook is to prepare a clean dataset showing where and when Cameroonian citizens acquired citizenship in European destination countries.

This source is not used to measure new migration entries. It captures a later administrative event that may occur after several years of residence in a destination country.

## 2. Analytical Role in the Project

This dataset supports the third research question of the project:

**Q3: How did post-arrival trajectory indicators evolve across the available 2015-2024 data period?**

Citizenship acquisition is used as an administrative indicator of long-term integration or settlement in a destination country.

It should be interpreted separately from:

- first residence permits
- long-term resident status
- status changes
- migrant stock

Each indicator measures a different phenomenon.

## 3. Source File Used

Raw input file:

```text
data/raw/europe/eurostat_citizenship_acquisition_cameroon.xlsx
```

Cleaned output file:

```text
data/processed/cleaned/eurostat_citizenship_acquisition_cleaned.csv
```

The raw file is loaded from the `data/raw/europe/` folder. The cleaned output is exported to the `data/processed/cleaned/` folder for later integration into the post-arrival analytical table.

## 4. Environment and Path Setup

This section imports the required Python libraries and defines the main project paths.

The notebook uses:

- `pandas` for loading, cleaning and reshaping the data
- `Path` from `pathlib` to manage file paths in a cleaner and more reproducible way

The output folder is also created automatically if it does not already exist.

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

EUROPE_PATH = DATA_RAW / "europe"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

## 5. Data Loading and Sheet Inspection

The Excel file is opened with `pd.ExcelFile()` in order to inspect the available sheets.

This step is important because Eurostat Excel files often contain metadata, notes, empty rows and non-standard headers before the actual data table.

In [2]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_citizenship_acquisition_cameroon.xlsx")
xls.sheet_names

c:\Users\Linno\OneDrive\Documents\my_projects\cameroonian-migration-analysis\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['Summary',
 'Sheet 1',
 'Sheet 2',
 'Sheet 3',
 'Sheet 4',
 'Sheet 5',
 'Sheet 6',
 'Sheet 7',
 'Sheet 8',
 'Sheet 9',
 'Sheet 10',
 'Sheet 11',
 'Sheet 12',
 'Sheet 13',
 'Sheet 14',
 'Sheet 15',
 'Sheet 16',
 'Sheet 17',
 'Sheet 18',
 'Sheet 19',
 'Sheet 20',
 'Sheet 21',
 'Sheet 22',
 'Sheet 23',
 'Sheet 24',
 'Sheet 25',
 'Sheet 26',
 'Sheet 27',
 'Sheet 28',
 'Sheet 29',
 'Sheet 30',
 'Sheet 31',
 'Sheet 32',
 'Sheet 33',
 'Sheet 34',
 'Sheet 35',
 'Sheet 36',
 'Sheet 37',
 'Sheet 38',
 'Sheet 39',
 'Sheet 40',
 'Sheet 41',
 'Sheet 42',
 'Sheet 43',
 'Sheet 44',
 'Sheet 45',
 'Sheet 46',
 'Sheet 47',
 'Sheet 48',
 'Sheet 49',
 'Sheet 50',
 'Sheet 51',
 'Sheet 52',
 'Sheet 53',
 'Sheet 54',
 'Sheet 55',
 'Sheet 56',
 'Sheet 57',
 'Sheet 58',
 'Sheet 59',
 'Sheet 60',
 'Sheet 61',
 'Sheet 62',
 'Sheet 63',
 'Sheet 64',
 'Sheet 65',
 'Sheet 66',
 'Sheet 67',
 'Sheet 68',
 'Sheet 69',
 'Sheet 70',
 'Sheet 71',
 'Sheet 72',
 'Sheet 73',
 'Sheet 74',
 'Sheet 75',
 'Sheet 76',
 'Sheet 7

## 6. Raw Data Inspection

The selected sheet is loaded without predefined headers.

This allows the notebook to inspect the original structure of the file before deciding which rows and columns should be kept or removed.

The goal is to identify:

- metadata rows
- empty or unnecessary columns
- the row containing the real header
- destination country labels
- year columns
- citizenship acquisition values

In [3]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_citizenship_acquisition_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

sheet1_raw

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,Data extracted on 09/04/2026 14:08:47 from [ES...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dataset:,"Acquisition of citizenship by age group, sex a...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Last updated:,27/03/2026 11:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Time frequency,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,Observation flags:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,b,break in time series,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,e,estimated,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,ep,"estimated, provisional",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Removing Metadata Rows and Unnecessary Columns

This section removes rows and columns that are not part of the analytical dataset.

The raw Eurostat file contains metadata, explanatory rows and spacing columns that are useful for human reading but not for analysis.

The cleaning keeps only the rows and columns needed to build a country-year dataset.

In [4]:
sheet1_raw = sheet1_raw.drop(columns=[2, 4, 6, 8, 10, 12, 14, 16, 18], errors="ignore")
sheet1_raw = sheet1_raw.drop(index=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 57, 58, 59, 60, 61, 62, 63, 64], errors="ignore").reset_index(drop=True)

sheet1_raw

,0,1,3,5,7,9,11,13,15,17,19
0,TIME,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
1,Belgium,738,845,872,955,1046,945,1196,1266,1311,1454
2,Bulgaria,0,0,1,0,0,1,0,0,0,1
3,Czechia,0,0,2,0,0,1,0,1,0,1
4,Denmark,14,51,14,3,2,18,26,20,15,50
5,Germany,1081,920,953,910,955,895,850,930,990,1210
6,Estonia,0,0,0,0,0,1,0,0,2,2
7,Ireland,79,43,38,33,27,24,46,57,58,75
8,Greece,0,3,3,1,0,0,5,1,1,2
9,Spain,135,196,83,123,152,258,241,357,348,351


## 8. Additional Row Cleaning

After the first cleaning pass, an additional row is removed because it does not correspond to a destination country observation.

This kind of manual cleaning decision is common when working with institutional Excel files that include notes, totals or formatting rows inside the table.

In [5]:
sheet1_raw = sheet1_raw.drop(index=[41], errors="ignore").reset_index(drop=True)

sheet1_raw

,0,1,3,5,7,9,11,13,15,17,19
0,TIME,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
1,Belgium,738,845,872,955,1046,945,1196,1266,1311,1454
2,Bulgaria,0,0,1,0,0,1,0,0,0,1
3,Czechia,0,0,2,0,0,1,0,1,0,1
4,Denmark,14,51,14,3,2,18,26,20,15,50
5,Germany,1081,920,953,910,955,895,850,930,990,1210
6,Estonia,0,0,0,0,0,1,0,0,2,2
7,Ireland,79,43,38,33,27,24,46,57,58,75
8,Greece,0,3,3,1,0,0,5,1,1,2
9,Spain,135,196,83,123,152,258,241,357,348,351


## 9. Handling Missing or Confidential Values

Eurostat files may use symbols such as `:` to indicate missing, unavailable or confidential values.

In this notebook, these values are replaced before the dataset is reshaped.

This decision should be interpreted carefully: replacing `:` with `0` makes the dataset easier to process, but it does not necessarily mean that the true value is zero. This is a methodological limitation to keep in mind when interpreting the results.

In [6]:
sheet1_raw = sheet1_raw.replace(":", 0)
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)

sheet1_raw

,0,1,3,5,7,9,11,13,15,17,19
0,TIME,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
1,Belgium,738,845,872,955,1046,945,1196,1266,1311,1454
2,Bulgaria,0,0,1,0,0,1,0,0,0,1
3,Czechia,0,0,2,0,0,1,0,1,0,1
4,Denmark,14,51,14,3,2,18,26,20,15,50
5,Germany,1081,920,953,910,955,895,850,930,990,1210
6,Estonia,0,0,0,0,0,1,0,0,2,2
7,Ireland,79,43,38,33,27,24,46,57,58,75
8,Greece,0,3,3,1,0,0,5,1,1,2
9,Spain,135,196,83,123,152,258,241,357,348,351


## 10. Header Standardization and Reshaping

The first valid row is used as the column header.

The dataset is then reshaped from wide format to long format.

The long format is more suitable for analysis because each row represents one observation defined by:

- destination country
- year
- citizenship acquisition value

This structure is easier to filter, aggregate, visualize and combine with other post-arrival indicators.

In [7]:
# Use the first row as the header
sheet1_raw.columns = sheet1_raw.iloc[0]
sheet1_raw = sheet1_raw.iloc[1:].reset_index(drop=True)

# Rename TIME to destination_country
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)

# Remove the GEO (Labels) row if it is still present
sheet1_raw = sheet1_raw[
    sheet1_raw["destination_country"] != "GEO (Labels)"
]

# Reshape from wide format to long format
sheet1_raw = sheet1_raw.melt(
    id_vars=["destination_country"],
    var_name="year",
    value_name="permit"
)

# Clean data types
sheet1_raw["year"] = sheet1_raw["year"].astype(int)

sheet1_raw["permit"] = (
    pd.to_numeric(sheet1_raw["permit"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# Reorder columns
sheet1_raw = sheet1_raw[
    ["destination_country", "year", "permit"]
]

sheet1_raw

,destination_country,year,permit
0,Belgium,2015,738
1,Bulgaria,2015,0
2,Czechia,2015,0
3,Denmark,2015,14
4,Germany,2015,1081
...,...,...,...
395,Türkiye,2024,0
396,Ukraine,2024,0
397,Belarus,2024,0
398,Russia,2024,0


## 11. Standardization of Output Columns

This section standardizes the cleaned dataset by renaming columns and adding metadata fields.

The added fields help keep the source and indicator type explicit:

- `source`: identifies Eurostat as the data source
- `nature`: identifies the indicator as citizenship acquisition

This makes the file easier to integrate later into the project-wide post-arrival table.

In [8]:
# Rename permit to permits
sheet1_raw = sheet1_raw.rename(
    columns={"permit": "permits"}
)

# Add source and nature columns
sheet1_raw["source"] = "eurostat"
sheet1_raw["nature"] = "citizenship_acquisition"

# Reorder columns
sheet1_raw = sheet1_raw[
    [
        "destination_country",
        "year",
        "permits",
        "source",
        "nature",
    ]
]

sheet1_raw

,destination_country,year,permits,source,nature
0,Belgium,2015,738,eurostat,citizenship_acquisition
1,Bulgaria,2015,0,eurostat,citizenship_acquisition
2,Czechia,2015,0,eurostat,citizenship_acquisition
3,Denmark,2015,14,eurostat,citizenship_acquisition
4,Germany,2015,1081,eurostat,citizenship_acquisition
...,...,...,...,...,...
395,Türkiye,2024,0,eurostat,citizenship_acquisition
396,Ukraine,2024,0,eurostat,citizenship_acquisition
397,Belarus,2024,0,eurostat,citizenship_acquisition
398,Russia,2024,0,eurostat,citizenship_acquisition


## 12. Output Export

The cleaned dataset is exported as a CSV file to:

```text
data/processed/cleaned/eurostat_citizenship_acquisition_cleaned.csv
```

This output will be used later to build or enrich the `post_arrival_master.csv` analytical table.

In [9]:
sheet1_raw.to_csv(
    CLEANED_PATH / "eurostat_citizenship_acquisition_cleaned.csv",
    index=False
)

## 13. Methodological Notes and Limitations

Citizenship acquisition should not be interpreted as migration entry.

It measures a later administrative event that can happen several years after arrival in the destination country.

Main limitations:

- it does not track individual migrant journeys
- it does not show when the person first arrived
- it does not represent the full Cameroonian migrant population abroad
- it covers only countries available in the Eurostat dataset
- missing or unavailable values may affect comparability across countries and years

For this reason, citizenship acquisition is treated as one post-arrival indicator, not as a complete measure of migration dynamics.

## 14. Conclusion

This notebook produces a cleaned Eurostat citizenship acquisition dataset for Cameroonian citizens.

The cleaned file is ready for the next stages of the pipeline, especially the construction of post-arrival analytical tables and the exploratory analysis of long-term administrative trajectories.